In [11]:
import os
import sys
import joblib
import numpy as np
import pandas as pd
sys.path.append("/dss/work/rirg2545/actionable-hypotension/")
from utils.db_interface import get_engine
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sqlalchemy import create_engine
from baseline_model.utils.calibrated_logreg_model import CalibratedLogRegModel
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

### Config

In [3]:
UNCALIBRATED_MODELS_DIR = "/dss/work/rirg2545/actionable-hypotension/extended_evaluation_review/baseline_extended/models/uncalibrated"
CALIBRATED_MODELS_DIR = "/dss/work/rirg2545/actionable-hypotension/extended_evaluation_review/baseline_extended/models/calibrated"
DATABASE_URI = "postgresql+psycopg2://rirg2545@localhost:5434/mimic"
engine = create_engine(DATABASE_URI, future=True)

### Setup

In [4]:
os.makedirs(UNCALIBRATED_MODELS_DIR, exist_ok=True)
os.makedirs(CALIBRATED_MODELS_DIR, exist_ok=True)

### Help Functions

In [9]:
def load_and_prepare_data_baseline():
    query = """
    SELECT
        mw.subject_id,
        mw.mean,
        mw.median,
        mw.min,
        mw.max,
        mw.std,
        mw.iqr,
        mw.first,
        mw.last,
        mw.slope,
        mw.rate_change,
        mw.weighted_mean,
        mw.positive_event,
        mw.split
    FROM ce_approach.merged_mix_features mw
    WHERE mw.split IN ('train', 'val', 'test')
    """
    df = pd.read_sql(query, engine)
    df["label"] = df["positive_event"].astype(int)

    all_map_features = ["mean", "median", "min", "max", "std", 
                        "iqr", "first", "last", "slope", 
                        "rate_change", "weighted_mean"]

    train = df[df["split"] == "train"]
    val   = df[df["split"] == "val"]
    test  = df[df["split"] == "test"]

    # Baseline 1: last MAP only (original)
    X_train_last = train[["last"]]
    X_val_last   = val[["last"]]
    X_test_last  = test[["last"]]

    # Baseline 2: all MAP features, logistic regression
    X_train_all  = train[all_map_features]
    X_val_all    = val[all_map_features]
    X_test_all   = test[all_map_features]

    y_train = train["label"]
    y_val   = val["label"]
    y_test  = test["label"]

    return (X_train_last, X_train_all,
            X_val_last,   X_val_all,
            X_test_last,  X_test_all,
            y_train, y_val, y_test)

In [13]:
def calibrate_logreg_isotonic(model, X_val, y_val):
    """
    Fit Isotonic Regression to calibrate Logistic Regression probabilities.
    """
    # Get predicted probabilities for validation set
    y_val_pred = model.predict_proba(X_val)[:, 1]

    # Sort predictions (required for isotonic regression)
    sorted_idx = np.argsort(y_val_pred)

    # Fit isotonic calibrator
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(y_val_pred[sorted_idx], y_val.iloc[sorted_idx])
    return iso

### Train Baseline Model

In [25]:
"""
(X_train_last, X_train_all,
 X_val_last,   X_val_all,
 X_test_last,  X_test_all,
 y_train, y_val, y_test) = load_and_prepare_data_baseline()

# Baseline 1: last MAP only
lr_last = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000))
])
lr_last.fit(X_train_last, y_train)
joblib.dump(lr_last, os.path.join(UNCALIBRATED_MODELS_DIR, "baseline_lr_last.pkl"))

# Baseline 2: all MAP features
lr_all = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000))
])
lr_all.fit(X_train_all, y_train)
joblib.dump(lr_all, os.path.join(UNCALIBRATED_MODELS_DIR, "baseline_lr_all_map.pkl"))
"""
# Baseline 3: MAP threshold rule (no training needed)
threshold = 65
y_pred_threshold = (X_test_last["last"] < threshold).astype(int)

### Calibrate Baseline Model

In [15]:
# Baseline 1: last MAP only
iso_last = calibrate_logreg_isotonic(lr_last, X_val_last, y_val)
calibrated_last = CalibratedLogRegModel(lr_last, iso_last)
joblib.dump(calibrated_last, os.path.join(CALIBRATED_MODELS_DIR, "baseline_lr_last.pkl"))

# Baseline 2: all MAP features
iso_all = calibrate_logreg_isotonic(lr_all, X_val_all, y_val)
calibrated_all = CalibratedLogRegModel(lr_all, iso_all)
joblib.dump(calibrated_all, os.path.join(CALIBRATED_MODELS_DIR, "baseline_lr_all_map.pkl"))

['/dss/work/rirg2545/actionable-hypotension/extended_evaluation_review/baseline_extended/models/calibrated/baseline_lr_all_map.pkl']

In [ ]:
# Evaluate baseline 1-
from sklearn.metrics import roc_auc_score
import numpy as np

def bootstrap_auc(y_true, y_prob, n_rounds=1000, seed=42):
    rng = np.random.default_rng(seed)
    aucs = []
    n = len(y_true)
    for _ in range(n_rounds):
        idx = rng.integers(0, n, size=n)
        if y_true[idx].nunique() < 2:
            continue  # skip rounds with only one class
        aucs.append(roc_auc_score(y_true.iloc[idx], y_prob[idx]))
    aucs = np.array(aucs)
    return {
        "auc": roc_auc_score(y_true, y_prob),
        "ci_lower": np.percentile(aucs, 2.5),
        "ci_upper": np.percentile(aucs, 97.5)
    }

y_test_reset = y_test.reset_index(drop=True)
prob_last = calibrated_last.predict_proba(X_test_last)[:, 1]
prob_all  = calibrated_all.predict_proba(X_test_all)[:, 1]

results_last = bootstrap_auc(y_test_reset, prob_last)
results_all  = bootstrap_auc(y_test_reset, prob_all)

print(f"Baseline (last MAP):   AUC {results_last['auc']:.3f} "
      f"(95% CI {results_last['ci_lower']:.3f}–{results_last['ci_upper']:.3f})")
print(f"Baseline (all MAP LR): AUC {results_all['auc']:.3f} "
      f"(95% CI {results_all['ci_lower']:.3f}–{results_all['ci_upper']:.3f})")

Baseline (last MAP):   AUC 0.685 (95% CI 0.673–0.698)
Baseline (all MAP LR): AUC 0.747 (95% CI 0.736–0.758)


In [23]:
from sklearn.metrics import roc_curve, confusion_matrix

def get_threshold_at_sensitivity(y_true, y_prob, min_sensitivity=0.80):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    # thresholds are descending, tpr ascending
    # find highest threshold that still achieves min_sensitivity
    valid_mask = tpr >= min_sensitivity
    if not valid_mask.any():
        return thresholds[-1]
    return thresholds[valid_mask][0]  # first (highest) threshold meeting criterion

def evaluate_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "threshold": threshold,
        "sensitivity": tp / (tp + fn),
        "specificity": tn / (tn + fp),
        "ppv":         tp / (tp + fp) if (tp + fp) > 0 else 0,
        "npv":         tn / (tn + fn) if (tn + fn) > 0 else 0,
    }

y_true = y_test_reset.values

for name, prob in [("last MAP", prob_last), ("all MAP LR", prob_all)]:
    thresh = get_threshold_at_sensitivity(y_true, prob, min_sensitivity=0.80)
    metrics = evaluate_at_threshold(y_true, prob, thresh)
    #results = bootstrap_auc(y_test_reset, prob)
    print(f"\nBaseline ({name}):")
    #print(f"  AUC:         {results['auc']:.3f} (95% CI {results['ci_lower']:.3f}–{results['ci_upper']:.3f})")
    print(f"  Threshold:   {metrics['threshold']}")
    print(f"  Sensitivity: {metrics['sensitivity']}")
    print(f"  Specificity: {metrics['specificity']}")
    print(f"  PPV:         {metrics['ppv']}")
    print(f"  NPV:         {metrics['npv']}")




Baseline (last MAP):
  Threshold:   0.0010463081557916764
  Sensitivity: 0.8110307414104883
  Specificity: 0.39007628927802585
  PPV:         0.002633591652365902
  NPV:         0.9990389280138688

Baseline (all MAP LR):
  Threshold:   0.0011560008299493143
  Sensitivity: 0.8037974683544303
  Specificity: 0.5302015049581746
  PPV:         0.0033860539521419933
  NPV:         0.9992656949250044


In [29]:
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix

y_true = y_test_reset.values
y_pred = (X_test_last["last"] < threshold).astype(int).values

# Standard classification metrics
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
ppv = tp / (tp + fp)
npv=    tn / (tn + fn) 

print(f"Threshold rule (LAST MAP < {threshold}):")
print(f"  Sensitivity: {sensitivity:}")
print(f"  Specificity: {specificity:}")
print(f"  PPV:         {ppv:}")
print(f"  NPV:         {npv:}")

Threshold rule (LAST MAP < 65):
  Sensitivity: 0.43896925858951175
  Specificity: 0.8185638440172437
  PPV:         0.004781461127853611
  NPV:         0.9986408251866812
